# Full-Data Backtesting and Model Selection Plan


## 1. Objective of This Notebook

The previous notebooks in this project focused on:

- defining the forecasting problem
- exploring the Gold feature mart
- building baseline features
- training initial benchmark models on a **subset of the dataset**

Those steps established a working modeling pipeline but **do not yet represent a production-grade evaluation**.

The purpose of this notebook is to transition from **subset experimentation** to **full-data model selection**.

This notebook defines the framework for:

- training models on the **full M5 dataset**
- evaluating models using **consistent backtesting**
- comparing multiple candidate models fairly
- determining the **best candidate model for Production v1**

Importantly, this notebook does **not yet deploy the final model**.  
Instead, it defines the **evaluation process used to choose the production model** that will later be operationalized in the MLOps pipeline.

## 2. Current State of the Modeling Workflow

Before defining the full-data backtesting plan, this notebook must clearly document the current modeling state.

### Current modeling data source

The modeling workflow currently reads from:

`gold.gold_m5_daily_feature_mart`

This table provides the stable daily modeling base at the grain:

- one row per `(store_id, item_id, date)`

The table currently includes:

- identifiers and product hierarchy columns
- date and retail week
- target variable: `sales`
- price variable: `sell_price`
- weather variables
- macroeconomic variables
- Google Trends variables

Importantly, this table does **not** yet contain notebook-engineered forecasting features such as:

- calendar breakdown fields
- lag features
- rolling mean features

Those were created later inside the modeling notebook.

### Current advanced-model workflow

In `04_advanced_model_candidate.ipynb`, the workflow:

1. selected the **top 20 series by total sales**
2. loaded those rows from `gold.gold_m5_daily_feature_mart`
3. engineered forecasting features in the notebook itself:
   - `day_of_week`
   - `day_of_month`
   - `week_of_year`
   - `month`
   - `lag_1`
   - `lag_7`
   - `lag_28`
   - `rolling_mean_7`
   - `rolling_mean_28`
4. used a **single 28-day holdout split**
5. trained and compared candidate models using **WMAPE**

### Current benchmark results

The current benchmark comparison from the earlier notebooks is:

| Model | WMAPE |
|---|---:|
| Naive lag-1 | 0.2893 |
| Rolling mean (7-day) | 0.2621 |
| Seasonal naive lag-7 | 0.2423 |
| Gradient Boosting | 0.2303 |
| Random Forest | **0.2195** |

### Why this is not yet production-grade model selection

The current workflow is useful, but it is still an early benchmark only.

It is not yet sufficient for production model selection because:

- it uses a **top-20 subset**, not the full dataset
- it relies on **one holdout split**, not rolling backtesting
- it evaluates mainly with **WMAPE**
- external variables are present in the source table, but they have **not yet been tested in a controlled ablation framework**
- feature engineering still lives partly in notebooks rather than entirely in a locked evaluation framework

Therefore, this notebook will now define the proper **full-data backtesting and model selection process** that must be completed before choosing the Production v1 model.

In [3]:
# ============================================================
# Notebook path setup
# ============================================================
# Purpose:
# - make the project root importable from inside the notebooks folder
# - allow imports such as `from training.dataset import ...`
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:")
print(PROJECT_ROOT)

Project root added to sys.path:
/Users/AKB_CIM/DS_HQ/projects/global-demand-forecasting


In [2]:
# ============================================================
# Imports for full-data backtesting plan
# ============================================================
# Purpose:
# - reuse the existing training package instead of rewriting logic
# - inspect the current reusable ML training building blocks
# - prepare this notebook to move from subset-based training
#   toward full-data-capable workflow design
# ============================================================

import warnings

import numpy as np
import pandas as pd

from sqlalchemy import text

from app_config.config import settings
from database.database import warehouse_engine

from training.dataset import load_full_modeling_dataset, load_top_series_subset
from training.evaluate import wmape
from training.features import FEATURE_COLUMNS, GROUP_COLS, build_features, prepare_modeling_dataset

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 200)

print("Imports loaded successfully.")
print("Warehouse DSN present:", bool(settings.WAREHOUSE_DSN or settings.POSTGRES_DSN))
print("Current feature columns:", FEATURE_COLUMNS)
print("Current grouping columns:", GROUP_COLS)
print("Current MAX_D_COLS:", settings.MAX_D_COLS)

Imports loaded successfully.
Warehouse DSN present: True
Current feature columns: ['day_of_week', 'day_of_month', 'week_of_year', 'month', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']
Current grouping columns: ['store_id', 'item_id']
Current MAX_D_COLS: 1913


## Full-data training design decision

This notebook will **not** train the full M5 dataset directly in local notebook memory.

Reason:

- the Gold mart contains **58,327,370 rows**
- repeated full-table extraction into pandas is too slow and too fragile
- that pattern is not suitable for production-grade model training

Therefore, the correct approach for the next stage is:

1. use this notebook to define the **full-data backtesting and model-selection framework**
2. keep notebook execution limited to:
   - warehouse profiling
   - small validation slices
   - backtesting design
   - feature-set planning
   - model-selection rules
3. move full-data model training into a **dedicated training pipeline**
4. run that pipeline in **AWS compute**, not in the notebook
5. stage modeling datasets through a **materialized extract / parquet-based training input**, rather than repeated ad hoc notebook pulls from the warehouse

Production direction for this project:

- notebook = research, design, and validation
- training package = reusable training code
- AWS job = full-data model training
- MLflow = experiment tracking and model registry
- MWAA = orchestration
- warehouse = source + forecast output storage

This means notebook 05 now focuses on defining the **production-grade full-data training plan**, not brute-force local training.

In [3]:
# ============================================================
# Confirm full-data warehouse scope without loading full data
# ============================================================
# Purpose:
# - ground notebook 05 in the actual production dataset scale
# - avoid pulling the full mart into pandas
# - make the next design decisions evidence-based
# ============================================================

warehouse_scope_query = """
select
    count(*) as row_count,
    count(distinct store_id || '_' || item_id) as series_count,
    count(distinct item_id) as item_count,
    count(distinct dept_id) as dept_count,
    count(distinct cat_id) as cat_count,
    count(distinct store_id) as store_count,
    count(distinct state_id) as state_count,
    min(date) as min_date,
    max(date) as max_date
from gold.gold_m5_daily_feature_mart
"""

with warehouse_engine.begin() as conn:
    result = conn.execute(text(warehouse_scope_query))
    warehouse_scope_df = pd.DataFrame(result.fetchall(), columns=result.keys())

warehouse_scope_df

,row_count,series_count,item_count,dept_count,cat_count,store_count,state_count,min_date,max_date
0,58327370,30490,3049,7,3,10,3,2011-01-29,2016-04-24


In [5]:
# ============================================================
# Inspect series distribution by store
# ============================================================
# Purpose:
# - determine the safest partitioning strategy for full-data training
# - confirm whether store-level partitioning is feasible
# - estimate how large each partition will be
# ============================================================

series_per_store_query = """
select
    store_id,
    count(distinct item_id) as item_count,
    count(*) as row_count
from gold.gold_m5_daily_feature_mart
group by store_id
order by store_id
"""

with warehouse_engine.begin() as conn:
    result = conn.execute(text(series_per_store_query))
    store_profile_df = pd.DataFrame(result.fetchall(), columns=result.keys())

store_profile_df

,store_id,item_count,row_count
0,CA_1,3049,5832737
1,CA_2,3049,5832737
2,CA_3,3049,5832737
3,CA_4,3049,5832737
4,TX_1,3049,5832737
5,TX_2,3049,5832737
6,TX_3,3049,5832737
7,WI_1,3049,5832737
8,WI_2,3049,5832737
9,WI_3,3049,5832737


In [4]:
# ============================================================
# Define the target production-grade training architecture
# ============================================================
# Purpose:
# - make the full-data training path explicit
# - separate notebook design work from production execution
# - lock the next implementation direction before writing more code
# ============================================================

training_architecture_df = pd.DataFrame(
    [
        {
            "layer": "Notebook 05",
            "role": "Design and validation",
            "responsibility": "Define backtesting rules, feature-set experiments, and model-selection criteria.",
        },
        {
            "layer": "Warehouse extract step",
            "role": "Training data preparation",
            "responsibility": "Materialize model-ready training slices from gold.gold_m5_daily_feature_mart.",
        },
        {
            "layer": "Object storage",
            "role": "Training input staging",
            "responsibility": "Store extracted training datasets as parquet for reliable and repeatable training runs.",
        },
        {
            "layer": "Training package",
            "role": "Reusable ML logic",
            "responsibility": "Load staged data, build features, train models, evaluate backtests, and log to MLflow.",
        },
        {
            "layer": "AWS training job",
            "role": "Full-data execution",
            "responsibility": "Run full-data model training outside the notebook on production-grade compute.",
        },
        {
            "layer": "MLflow",
            "role": "Experiment tracking and registry",
            "responsibility": "Track runs, metrics, artifacts, and promote the winning candidate.",
        },
        {
            "layer": "MWAA",
            "role": "Orchestration",
            "responsibility": "Schedule retraining and batch scoring after the modeling framework is finalized.",
        },
    ]
)

training_architecture_df

,layer,role,responsibility
0,Notebook 05,Design and validation,"Define backtesting rules, feature-set experime..."
1,Warehouse extract step,Training data preparation,Materialize model-ready training slices from g...
2,Object storage,Training input staging,Store extracted training datasets as parquet f...
3,Training package,Reusable ML logic,"Load staged data, build features, train models..."
4,AWS training job,Full-data execution,Run full-data model training outside the noteb...
5,MLflow,Experiment tracking and registry,"Track runs, metrics, artifacts, and promote th..."
6,MWAA,Orchestration,Schedule retraining and batch scoring after th...


## Decision: what notebook 05 will produce

Notebook 05 will produce the **design contract** for the next implementation phase.

Its outputs are:

- the evaluation framework for full-data model selection
- the backtesting structure to be used for candidate comparison
- the feature-set experiment plan, including external feature ablations
- the shortlist of production-candidate model families
- the target training architecture for full-data execution outside the notebook

Notebook 05 will therefore answer:

1. how models must be evaluated fairly
2. which feature sets must be compared
3. which model families advance to serious training
4. what full-data training pipeline must be built next

This keeps the notebook aligned with best practice:

- notebook for design and controlled validation
- production code for actual large-scale training

In [5]:
# ============================================================
# Define candidate feature-set experiments for full-data training
# ============================================================
# Purpose:
# - lock the feature ablation plan before building new training code
# - ensure external signals are tested systematically
# - prevent ad hoc feature selection later in the pipeline
# ============================================================

feature_experiments_df = pd.DataFrame(
    [
        {
            "experiment_id": "fs_01",
            "feature_set_name": "calendar_lag_rolling_baseline",
            "includes": "calendar + lag + rolling mean features",
            "purpose": "Reference baseline carried forward from earlier notebooks and training code.",
        },
        {
            "experiment_id": "fs_02",
            "feature_set_name": "baseline_plus_price",
            "includes": "calendar + lag + rolling mean + sell_price",
            "purpose": "Test whether price improves forecasting beyond historical demand features alone.",
        },
        {
            "experiment_id": "fs_03",
            "feature_set_name": "baseline_plus_weather",
            "includes": "calendar + lag + rolling mean + weather features",
            "purpose": "Test whether weather adds predictive value when modeled nonlinearly.",
        },
        {
            "experiment_id": "fs_04",
            "feature_set_name": "baseline_plus_macro",
            "includes": "calendar + lag + rolling mean + macro features",
            "purpose": "Test whether macroeconomic context improves performance.",
        },
        {
            "experiment_id": "fs_05",
            "feature_set_name": "baseline_plus_trends",
            "includes": "calendar + lag + rolling mean + Google Trends features",
            "purpose": "Test whether search-demand signals improve forecasts.",
        },
        {
            "experiment_id": "fs_06",
            "feature_set_name": "full_feature_set",
            "includes": "calendar + lag + rolling mean + price + weather + macro + Google Trends",
            "purpose": "Test the strongest broad feature set candidate for Production v1.",
        },
    ]
)

feature_experiments_df

,experiment_id,feature_set_name,includes,purpose
0,fs_01,calendar_lag_rolling_baseline,calendar + lag + rolling mean features,Reference baseline carried forward from earlie...
1,fs_02,baseline_plus_price,calendar + lag + rolling mean + sell_price,Test whether price improves forecasting beyond...
2,fs_03,baseline_plus_weather,calendar + lag + rolling mean + weather features,Test whether weather adds predictive value whe...
3,fs_04,baseline_plus_macro,calendar + lag + rolling mean + macro features,Test whether macroeconomic context improves pe...
4,fs_05,baseline_plus_trends,calendar + lag + rolling mean + Google Trends ...,Test whether search-demand signals improve for...
5,fs_06,full_feature_set,calendar + lag + rolling mean + price + weathe...,Test the strongest broad feature set candidate...


In [6]:
# ============================================================
# Define candidate model families for production-v1 selection
# ============================================================
# Purpose:
# - lock the next serious model families before implementation
# - keep model comparison disciplined and reproducible
# - align candidate selection with practical retail forecasting workflows
# ============================================================

model_candidates_df = pd.DataFrame(
    [
        {
            "model_id": "m_01",
            "model_family": "seasonal_naive_lag_7",
            "stage": "baseline",
            "interpretability": "high",
            "expected_role": "sanity-check benchmark",
        },
        {
            "model_id": "m_02",
            "model_family": "rolling_mean_baseline",
            "stage": "baseline",
            "interpretability": "high",
            "expected_role": "stronger simple benchmark",
        },
        {
            "model_id": "m_03",
            "model_family": "random_forest",
            "stage": "current_ml_benchmark",
            "interpretability": "medium",
            "expected_role": "carry-forward benchmark from current training package",
        },
        {
            "model_id": "m_04",
            "model_family": "lightgbm",
            "stage": "primary_advanced_candidate",
            "interpretability": "medium",
            "expected_role": "most likely Production v1 winner",
        },
        {
            "model_id": "m_05",
            "model_family": "xgboost",
            "stage": "secondary_advanced_candidate",
            "interpretability": "medium",
            "expected_role": "backup advanced tabular candidate",
        },
    ]
)

model_candidates_df

,model_id,model_family,stage,interpretability,expected_role
0,m_01,seasonal_naive_lag_7,baseline,high,sanity-check benchmark
1,m_02,rolling_mean_baseline,baseline,high,stronger simple benchmark
2,m_03,random_forest,current_ml_benchmark,medium,carry-forward benchmark from current training ...
3,m_04,lightgbm,primary_advanced_candidate,medium,most likely Production v1 winner
4,m_05,xgboost,secondary_advanced_candidate,medium,backup advanced tabular candidate


In [7]:
# ============================================================
# Define the backtesting strategy for model comparison
# ============================================================
# Purpose:
# - establish the evaluation protocol before training any new models
# - avoid single-split evaluation bias
# - align with time-series forecasting best practices
# ============================================================

backtesting_strategy_df = pd.DataFrame(
    [
        {
            "component": "forecast_horizon",
            "value": 28,
            "description": "28-day forecasting horizon matching the M5 competition setup.",
        },
        {
            "component": "evaluation_windows",
            "value": 5,
            "description": "Use 5 rolling backtest windows instead of a single holdout.",
        },
        {
            "component": "split_type",
            "value": "time_series_split",
            "description": "Maintain temporal ordering with no random shuffling.",
        },
        {
            "component": "primary_metric",
            "value": "WRMSSE",
            "description": "Competition-aligned metric for hierarchical demand forecasting.",
        },
        {
            "component": "secondary_metrics",
            "value": "WMAPE, MAE, RMSE",
            "description": "Supplementary metrics for operational monitoring and debugging.",
        },
    ]
)

backtesting_strategy_df

,component,value,description
0,forecast_horizon,28,28-day forecasting horizon matching the M5 com...
1,evaluation_windows,5,Use 5 rolling backtest windows instead of a si...
2,split_type,time_series_split,Maintain temporal ordering with no random shuf...
3,primary_metric,WRMSSE,Competition-aligned metric for hierarchical de...
4,secondary_metrics,"WMAPE, MAE, RMSE",Supplementary metrics for operational monitori...


In [8]:
# ============================================================
# Define model selection rules for Production v1
# ============================================================
# Purpose:
# - prevent subjective model choice later
# - make promotion criteria explicit before implementation
# - ensure the winning model is accurate, stable, and deployable
# ============================================================

model_selection_rules_df = pd.DataFrame(
    [
        {
            "rule_id": "r_01",
            "rule_name": "primary_metric_rule",
            "decision_rule": "The winning candidate must rank best on average WRMSSE across all backtest windows.",
        },
        {
            "rule_id": "r_02",
            "rule_name": "stability_rule",
            "decision_rule": "A model cannot win if it performs well on one window but degrades materially on others.",
        },
        {
            "rule_id": "r_03",
            "rule_name": "baseline_rule",
            "decision_rule": "A production candidate must beat the baseline models consistently, not marginally on one split.",
        },
        {
            "rule_id": "r_04",
            "rule_name": "operational_rule",
            "decision_rule": "A model must be trainable and scoreable within practical pipeline time and memory limits.",
        },
        {
            "rule_id": "r_05",
            "rule_name": "interpretability_rule",
            "decision_rule": "When accuracy is close, prefer the model that is easier to explain, monitor, and debug.",
        },
        {
            "rule_id": "r_06",
            "rule_name": "feature_value_rule",
            "decision_rule": "External features remain in scope only if ablation testing shows measurable and repeatable benefit.",
        },
    ]
)

model_selection_rules_df

,rule_id,rule_name,decision_rule
0,r_01,primary_metric_rule,The winning candidate must rank best on averag...
1,r_02,stability_rule,A model cannot win if it performs well on one ...
2,r_03,baseline_rule,A production candidate must beat the baseline ...
3,r_04,operational_rule,A model must be trainable and scoreable within...
4,r_05,interpretability_rule,"When accuracy is close, prefer the model that ..."
5,r_06,feature_value_rule,External features remain in scope only if abla...


In [9]:
# ============================================================
# Define the implementation roadmap after notebook 05
# ============================================================
# Purpose:
# - translate notebook decisions into the next build steps
# - make the handoff from research to production code explicit
# - keep future MLOps work aligned with the agreed modeling contract
# ============================================================

implementation_roadmap_df = pd.DataFrame(
    [
        {
            "step_no": 1,
            "next_component": "training.dataset",
            "required_change": "Add warehouse-side extract logic that materializes training datasets by backtest window and feature-set definition.",
        },
        {
            "step_no": 2,
            "next_component": "training.features",
            "required_change": "Refactor feature engineering so baseline, price, weather, macro, trends, and full feature sets can be selected by config.",
        },
        {
            "step_no": 3,
            "next_component": "training.evaluate",
            "required_change": "Add MAE, RMSE, and WRMSSE-ready evaluation utilities for rolling backtests.",
        },
        {
            "step_no": 4,
            "next_component": "training backtest runner",
            "required_change": "Create a reusable rolling backtesting entrypoint for candidate model comparison.",
        },
        {
            "step_no": 5,
            "next_component": "training model entrypoints",
            "required_change": "Add LightGBM and XGBoost training scripts alongside the existing Random Forest benchmark.",
        },
        {
            "step_no": 6,
            "next_component": "mlflow integration",
            "required_change": "Track all serious candidate runs with feature-set ID, backtest window, metrics, and artifacts.",
        },
        {
            "step_no": 7,
            "next_component": "AWS training job",
            "required_change": "Move full-data training execution out of the notebook and into production-grade compute.",
        },
    ]
)

implementation_roadmap_df

,step_no,next_component,required_change
0,1,training.dataset,Add warehouse-side extract logic that material...
1,2,training.features,"Refactor feature engineering so baseline, pric..."
2,3,training.evaluate,"Add MAE, RMSE, and WRMSSE-ready evaluation uti..."
3,4,training backtest runner,Create a reusable rolling backtesting entrypoi...
4,5,training model entrypoints,Add LightGBM and XGBoost training scripts alon...
5,6,mlflow integration,Track all serious candidate runs with feature-...
6,7,AWS training job,Move full-data training execution out of the n...


In [10]:
# ============================================================
# Collect notebook decision tables for later export and reuse
# ============================================================
# Purpose:
# - keep all design tables in one structured object
# - prepare clean export to Excel later
# - make notebook outputs easier for future data scientists to reuse
# ============================================================

decision_tables = {
    "training_architecture": training_architecture_df,
    "feature_experiments": feature_experiments_df,
    "model_candidates": model_candidates_df,
    "backtesting_strategy": backtesting_strategy_df,
    "model_selection_rules": model_selection_rules_df,
    "implementation_roadmap": implementation_roadmap_df,
}

print("Decision tables prepared for export:")
for name, df in decision_tables.items():
    print(f"- {name}: {df.shape[0]} rows x {df.shape[1]} columns")

Decision tables prepared for export:
- training_architecture: 7 rows x 3 columns
- feature_experiments: 6 rows x 4 columns
- model_candidates: 5 rows x 5 columns
- backtesting_strategy: 5 rows x 3 columns
- model_selection_rules: 6 rows x 3 columns
- implementation_roadmap: 7 rows x 3 columns


In [13]:
! pip install xlsxwriter

In [11]:
# ============================================================
# Export modeling design contract to Excel
# ============================================================
# Purpose:
# - create a structured workbook documenting modeling decisions
# - make the design easily accessible to future data scientists
# - provide a refreshable artifact for the MLOps workflow
# ============================================================

from pathlib import Path

output_dir = PROJECT_ROOT / "artifacts"
output_dir.mkdir(exist_ok=True)

excel_path = output_dir / "gdf_modeling_design_contract.xlsx"

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:

    for sheet_name, df in decision_tables.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        # Format as Excel table
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name]

        rows, cols = df.shape

        worksheet.add_table(
            0,
            0,
            rows,
            cols - 1,
            {
                "columns": [{"header": col} for col in df.columns]
            }
        )

        worksheet.set_column(0, cols - 1, 28)

print("Excel workbook created:")
print(excel_path)

Excel workbook created:
/Users/AKB_CIM/DS_HQ/projects/global-demand-forecasting/artifacts/gdf_modeling_design_contract.xlsx
